# 01 - EDA and Data Preparation

This notebook uses `resumeJD2_pairs.csv` and prepares the same downstream artifact expected by the original project: `cleaned_resumeJD_pairs.csv`.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_colwidth', 120)
sns.set_theme(style='whitegrid')

In [ ]:
# Load resumeJD2_pairs.csv.
# In Colab, either upload it when prompted or place it at /content/resumeJD2_pairs.csv.

DATA_PATH_CANDIDATES = [
    'resumeJD2_pairs.csv',
    'dataset/resumeJD2_pairs.csv',
    '/content/resumeJD2_pairs.csv',
    '/content/drive/MyDrive/resumeJD2_pairs.csv',
]

data_path = next((p for p in DATA_PATH_CANDIDATES if os.path.exists(p)), None)

if data_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        data_path = next(iter(uploaded.keys()))
    except Exception as exc:
        raise FileNotFoundError('Could not find resumeJD2_pairs.csv. Upload it or update DATA_PATH_CANDIDATES.') from exc

df = pd.read_csv(data_path, engine='python', on_bad_lines='skip')
print(f'Data Loaded Successfully: {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'Source: {data_path}')
print(f'Columns: {list(df.columns)}')

In [ ]:
# Standardize schema to match the original notebooks.
expected_cols = ['resume_text', 'job_description', 'match_score', 'match_label']
missing = [c for c in expected_cols if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

df = df[expected_cols].copy()
df.head()

In [ ]:
print('Dataset Info')
print('=' * 60)
df.info()

print('\nStatistical Summary')
display(df.describe(include='all'))

In [ ]:
missing_df = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Percentage': (df.isnull().sum() / len(df) * 100).round(3)
})
print('Missing Values Analysis')
print('=' * 60)
display(missing_df)
print(f'Total missing values: {df.isnull().sum().sum()}')

print(f'Full duplicate rows: {df.duplicated().sum()}')
print(f"Duplicate resume+JD pairs: {df.duplicated(subset=['resume_text','job_description']).sum()}")

In [ ]:
print('Unique labels found:')
print(df['match_label'].value_counts(dropna=False))

df['match_score'] = pd.to_numeric(df['match_score'], errors='coerce')
invalid_scores = df[df['match_score'].isna() | (df['match_score'] < 0) | (df['match_score'] > 1)]
print(f'Invalid scores (<0/>1/non-numeric): {len(invalid_scores)}')
display(invalid_scores.head(10))

In [ ]:
# Map resumeJD2 labels into the original project's high/medium/low contract.
label_map = {
    'match': 'high',
    'partial match': 'medium',
    'no match': 'low',
    'high': 'high',
    'medium': 'medium',
    'low': 'low',
}

df['match_label_original'] = df['match_label']
df['match_label'] = df['match_label'].astype(str).str.lower().str.strip().map(label_map)

unknown_labels = df[df['match_label'].isna()]['match_label_original'].value_counts()
if len(unknown_labels):
    print('Unknown labels found. Add them to label_map:')
    print(unknown_labels)
    raise ValueError('Unknown labels cannot be mapped to high/medium/low.')

print('Mapped label distribution:')
print(df['match_label'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
counts = df['match_label'].value_counts().reindex(['high', 'medium', 'low'])

axes[0].bar(counts.index, counts.values, color=['green', 'orange', 'red'])
axes[0].set_title('Label Distribution')
axes[0].set_ylabel('Count')

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%', colors=['green', 'orange', 'red'])
axes[1].set_title('Label Distribution')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['match_score'].dropna(), bins=30, color='steelblue', edgecolor='black')
axes[0].set_title('Score Distribution')
axes[0].set_xlabel('match_score')

sns.violinplot(data=df, x='match_label', y='match_score', order=['low', 'medium', 'high'], palette=['red', 'orange', 'green'], ax=axes[1])
axes[1].set_title('Score Distribution per Label')

plt.tight_layout()
plt.show()

In [ ]:
# Text length features used by later notebooks.
df['resume_len'] = df['resume_text'].astype(str).str.split().str.len()
df['jd_len'] = df['job_description'].astype(str).str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['resume_len'], bins=30, color='steelblue', edgecolor='black')
axes[0].set_title('Resume Word Count Distribution')
axes[0].set_xlabel('Words')

axes[1].hist(df['jd_len'], bins=30, color='coral', edgecolor='black')
axes[1].set_title('Job Description Word Count Distribution')
axes[1].set_xlabel('Words')

plt.tight_layout()
plt.show()

corr = df[['resume_len', 'jd_len', 'match_score']].corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.show()

In [ ]:
# Clean data while preserving the same column contract for Notebook 02/03.
df_clean = df.copy()
before = len(df_clean)

df_clean = df_clean.dropna(subset=['resume_text', 'job_description', 'match_score', 'match_label'])
df_clean = df_clean.drop_duplicates(subset=['resume_text', 'job_description'])
df_clean = df_clean[(df_clean['match_score'] >= 0) & (df_clean['match_score'] <= 1)]

df_clean['resume_len'] = df_clean['resume_text'].astype(str).str.split().str.len()
df_clean['jd_len'] = df_clean['job_description'].astype(str).str.split().str.len()

# Keep useful text only. These thresholds match the spirit of the original notebook.
df_clean = df_clean[(df_clean['resume_len'] >= 20) & (df_clean['jd_len'] >= 10)]

df_clean['match_label_lower'] = df_clean['match_label'].str.lower().str.strip()
df_clean = df_clean.drop(columns=['match_label_original'], errors='ignore')
df_clean = df_clean.reset_index(drop=True)

print('=' * 45)
print('CLEANING SUMMARY')
print('=' * 45)
print(f'Original rows:        {before}')
print(f'Final rows:           {len(df_clean)}')
print(f'Rows removed:         {before - len(df_clean)}')
print(f'Nulls remaining:      {df_clean.isnull().sum().sum()}')
print(f'Duplicates remaining: {df_clean.duplicated(subset=["resume_text", "job_description"]).sum()}')
print(f'Label variants:       {df_clean["match_label"].nunique()}')
print(f'Score range:          {df_clean["match_score"].min()} - {df_clean["match_score"].max()}')
print('\nLabel distribution after cleaning:')
print(df_clean['match_label'].value_counts())
display(df_clean.head(3))

In [ ]:
output_path = 'cleaned_resumeJD_pairs.csv'
df_clean.to_csv(output_path, index=False)
print(f'Saved: {output_path} ({len(df_clean)} rows)')

try:
    from google.colab import files
    files.download(output_path)
except Exception:
    pass